# EDA for BDD dataset 

For this EDA, we will focus on metadata only (which comes from the labels) as we are short on time.  One can also add image level EDA, but it will require more resources and time. Hence we are focusing on train and validation set only. 

A single datapoint from BDD dataset has following data 

1. Image 
2. Weather
3. Time of the Day
4. Timestamp 
5. List of Bounding boxes in the image
6. Whether the bounding box was occluded/truncated or not 
7. `manualshape` and `manualattributes` for bbox # TODO know more about this.


### Class distribution for Object detection classes

There are 10 classes in BDD dataset for object detection task, and in the paper they have given following distribution for the classes 

![image.png](../../assets/image.png)


We can notice the long tailed distribution for the classes and `train` and `motor` classes are under-represented. Hence, if model is not trained properly, model can perform bad for these class instances. Need to handle this imbalance. 

In [6]:
import json
import pandas as pd 
import pathlib 

In [22]:
class BDDMetadata:

    def __init__(self, file_path:pathlib.Path):
        self.raw_data = json.load(open(file_path,'r'))
        self.build_dataframes()
    
    def build_dataframes(self):
        img_rows = []
        object_rows = []

        for img in self.raw_data:
            img_name = img.get("name")

            img_rows.append({
                "image": img_name,
                "width": img.get("attributes", {}).get("resolution", [1280, 720])[0],
                "height": img.get("attributes", {}).get("resolution", [1280, 720])[1],
                "weather": img.get("attributes", {}).get("weather"),
                "timeofday": img.get("attributes", {}).get("timeofday"),
                "scene": img.get("attributes", {}).get("scene"),
            })

            if "labels" not in img: # nno bbox case 
                continue

            for bbox in img["labels"]:
                if "box2d" not in bbox:
                    continue

                box = bbox["box2d"]

                width = box["x2"] - box["x1"]
                height = box["y2"] - box["y1"]

                object_rows.append({
                    "image": img_name,
                    "category": bbox.get("category"),
                    "x1": box["x1"],
                    "y1": box["y1"],
                    "x2": box["x2"],
                    "y2": box["y2"],
                    "width": width,
                    "height": height,
                    "area": width * height,
                    "aspect_ratio": width / (height),
                    "occluded": bbox.get("attributes", {}).get("occluded"),
                    "truncated": bbox.get("attributes", {}).get("truncated"),
                })
        self.images_df = pd.DataFrame(img_rows)
        self.bbox_df = pd.DataFrame(object_rows)

In [24]:
# data= json.load(open('../../data/assignment_data_bdd/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json','r'))

dataset = BDDMetadata('../../data/assignment_data_bdd/bdd100k_labels_release/bdd100k/labels/bdd100k_labels_images_val.json')

In [25]:
dataset.images_df

,image,width,height,weather,timeofday,scene
0,b1c66a42-6f7d68ca.jpg,1280,720,overcast,daytime,city street
1,b1c81faa-3df17267.jpg,1280,720,clear,night,highway
2,b1c81faa-c80764c5.jpg,1280,720,clear,night,highway
3,b1c9c847-3bda4659.jpg,1280,720,undefined,daytime,city street
4,b1ca2e5d-84cf9134.jpg,1280,720,clear,night,city street
...,...,...,...,...,...,...
9995,fe1d9184-d144106a.jpg,1280,720,clear,daytime,city street
9996,fe1d9184-dec09b65.jpg,1280,720,overcast,daytime,city street
9997,fe1f2409-5b415eb7.jpg,1280,720,clear,daytime,residential
9998,fe1f2409-c16ea1ed.jpg,1280,720,clear,daytime,city street


In [26]:
dataset.bbox_df

,image,category,x1,y1,x2,y2,width,height,area,aspect_ratio,occluded,truncated
0,b1c66a42-6f7d68ca.jpg,traffic sign,1000.698742,281.992415,1040.626872,326.911560,39.928130,44.919145,1793.537461,0.888889,False,False
1,b1c66a42-6f7d68ca.jpg,traffic sign,214.613695,172.190058,274.505889,229.586743,59.892194,57.396685,3437.613393,1.043478,False,False
2,b1c66a42-6f7d68ca.jpg,traffic sign,797.314833,313.186265,829.756437,341.884608,32.441604,28.698343,931.020279,1.130435,False,False
3,b1c66a42-6f7d68ca.jpg,traffic sign,652.575363,303.204232,685.016968,315.681772,32.441605,12.477540,404.791424,2.600000,False,False
4,b1c66a42-6f7d68ca.jpg,traffic light,707.476543,311.938510,716.210821,328.159313,8.734278,16.220803,141.677003,0.538462,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...
185521,fe1f55fa-19ba3600.jpg,person,491.615094,388.051507,511.726280,444.314645,20.111186,56.263138,1131.518433,0.357449,False,False
185522,fe1f55fa-19ba3600.jpg,person,477.889798,361.848672,496.606109,408.015571,18.716311,46.166899,864.074040,0.405405,True,False
185523,fe1f55fa-19ba3600.jpg,person,583.948891,384.308246,611.399480,431.722899,27.450589,47.414653,1301.560152,0.578947,True,False
185524,fe1f55fa-19ba3600.jpg,person,405.520064,408.015571,415.502096,429.227390,9.982032,21.211819,211.737056,0.470588,False,False
